# Tema 3: Fundamentos de Arquitectura




# 3.1 ¿Qué es una Arquitectura de Agente?

## Definición Formal

> **Arquitectura de Agente:** El conjunto de decisiones estructurales que determinan cómo se organiza, comunica y coordina un sistema agéntico para lograr sus objetivos.

La arquitectura es la **estructura** del sistema - el esqueleto sobre el cual se construye todo lo demás.

## Las Tres Dimensiones del Diseño Agéntico

Es crucial entender que diseñar un sistema agéntico involucra **tres dimensiones distintas pero interrelacionadas**:

```
┌─────────────────────────────────────────────────────┐
│          LAS 3 DIMENSIONES DEL DISEÑO               │
└─────────────────────────────────────────────────────┘

1. ARQUITECTURA (Estructura)
   Pregunta: ¿Cómo está CONSTRUIDO el sistema?
   │
   ├─ Número de agentes (1 vs N)
   ├─ Topología (peer-to-peer, hierarchical)
   ├─ Boundaries (qué está dentro/fuera)
   └─ Puntos de integración
   
   Decisión: Design-time (antes de ejecutar)
   Ejemplo: "Usaremos 3 agentes especializados"

2. FLUJO (Comportamiento)
   Pregunta: ¿Cómo OPERA el sistema?
   │
   ├─ Secuencia de operaciones
   ├─ Control flow (loops, branches)
   ├─ Timing y sincronización
   └─ Patrones de orquestación
   
   Decisión: Run-time (durante ejecución)
   Ejemplo: "ReAct loop con reflection"

3. CAPACIDADES (Habilitadores)
   Pregunta: ¿Qué PUEDE HACER el sistema?
   │
   ├─ Tools disponibles
   ├─ Tipos de memoria
   ├─ Modelos de LLM
   └─ Integraciones externas
   
   Decisión: Configuration-time
   Ejemplo: "Tiene acceso a search + database"
```

| Dimensión | Pregunta Clave | Define | Cuándo se decide | Ejemplo |
|---|---|---|---|---|
| ARQUITECTURA | ¿Cómo está CONSTRUIDO? | Componentes, topología, boundaries | Design-time | Single-Agent vs Multi-Agent |
| FLUJO | ¿Cómo OPERA? | Secuencia, timing, control | Run-time | ReAct vs Plan-and-Execute |
| CAPACIDADES | ¿Qué PUEDE HACER? | Tools, memoria, integraciones | Configuration-time | Web search + Database access |

### Relación Entre Dimensiones

**Importante:** Estas dimensiones **NO son independientes**.

```
ARQUITECTURA habilita ciertos FLUJOS
CAPACIDADES determinan qué ARQUITECTURAS son viables
FLUJO aprovecha las CAPACIDADES disponibles
```

**Ejemplos de interdependencia:**

1. **Arquitectura → Flujo:**
   - Multi-Agent hierarchical → HABILITA flujos con delegación
   - Single-Agent → NO puede hacer "debate entre agentes"

2. **Capacidades → Arquitectura:**
   - Sin tools → Single-Agent suficiente
   - Con 100+ tools especializadas → Multi-Agent

3. **Flujo → Capacidades:**
   - Reflection pattern → REQUIERE capacidad de self-critique
   - Parallel execution → REQUIERE múltiples instancias de LLM

## Decisiones Estructurales Clave

Al diseñar una arquitectura de agente, debes decidir:

### 1. Número de Agentes

**Decisión fundamental:** ¿Uno o múltiples agentes?

```
Single-Agent:           Multi-Agent:
    ┌──────┐               ┌──────┐
    │Agent │               │Agent1│
    │      │               └──┬───┘
    │ LLM  │                  │
    │Tools │          ┌───────┼───────┐
    └──────┘          │       │       │
                   ┌──▼──┐ ┌──▼──┐ ┌──▼──┐
                   │Ag 2 │ │Ag 3 │ │Ag 4 │
                   └─────┘ └─────┘ └─────┘
```

### 2. Topología de Comunicación

Si Multi-Agent, ¿cómo se comunican?

```
Peer-to-Peer:           Hierarchical:           Hub-and-Spoke:
  A ←→ B                    Manager                  Coordinator
  ↕ ⤫ ↕                       ↓                      ↙  ↓  ↘
  C ←→ D                  A ← B → C               A   B   C
```

### 3. Puntos de Integración

¿Dónde y cómo se conecta el sistema con el mundo exterior?

- Input: ¿API, UI, webhook, queue?
- Output: ¿Callback, streaming, batch?
- Tools: ¿APIs externas, databases, code execution?

### 4. Boundaries del Sistema

¿Qué está dentro vs fuera del sistema agéntico?

```
┌─────────────────────────────────┐
│   DENTRO del Sistema Agéntico   │
│                                 │
│  • Agente(s)                    │
│  • Memoria interna              │
│  • Orquestación                 │
│  • State management             │
└─────────────────────────────────┘
          ↕ (boundary)
┌─────────────────────────────────┐
│   FUERA del Sistema Agéntico    │
│                                 │
│  • Vector DB (RAG)              │
│  • External APIs                │
│  • User interface               │
│  • Persistent storage           │
└─────────────────────────────────┘
```

## Arquitectura como Restricción Habilitadora

**Concepto clave:** La arquitectura **restringe** para **habilitar**.

### Restricción ≠ Limitación

Una buena arquitectura:
- ✅ Hace **imposible** algunos diseños malos
- ✅ Hace **fácil** los diseños correctos
- ✅ Reduce el espacio de decisiones

**Analogía:** Como un molde para hornear
- El molde "restringe" la forma del pastel
- Pero esa restricción "habilita" que salga con la forma deseada
- Sin molde: más libertad pero peor resultado

### Ejemplo Concreto

**Decisión arquitectónica:** "Usaremos Single-Agent"

**Restricciones que impone:**
- ❌ No puedes hacer "debate entre agentes" A2A
- ❌ No puedes especializar por rol
- ❌ No hay paralelización real

**Capacidades que habilita:**
- ✅ Contexto unificado (no necesitas sincronizar)
- ✅ Debugging simple (un solo punto de fallo)
- ✅ Menor latencia (una sola llamada LLM)
- ✅ Costo predecible (un solo modelo)

**La restricción hace posible** optimizaciones que serían imposibles en Multi-Agent.


# 3.2 Single-Agent Architecture

## Descripción

**Single-Agent:** Un único agente con un único LLM que coordina todas las capacidades del sistema.

```
┌─────────────────────────────────────────┐
│         SINGLE-AGENT SYSTEM             │
└─────────────────────────────────────────┘
                    │
                    │ User Input
                    ▼
         ┌──────────────────────┐
         │                      │
         │   AGENT (LLM Core)   │ ← UN SOLO CEREBRO
         │                      │
         │  • Reasoning         │
         │  • Planning          │
         │  • Decision making   │
         │                      │
         └──────────┬───────────┘
                    │
        ┌───────────┼───────────┐
        │           │           │
        ▼           ▼           ▼
    ┌──────┐   ┌──────┐   ┌──────┐
    │Tool 1│   │Tool 2│   │Tool N│ ← MÚLTIPLES HERRAMIENTAS
    └──────┘   └──────┘   └──────┘
        │           │           │
        └───────────┴───────────┘
                    │
         ┌──────────▼───────────┐
         │   UNIFIED CONTEXT    │ ← CONTEXTO UNIFICADO
         │   • Working memory   │
         │   • Episodic memory  │
         │   • State            │
         └──────────────────────┘
```

## Características Fundamentales

### 1. Un Solo LLM (Cerebro)

- **Todo el razonamiento** pasa por un único modelo
- No hay "delegación" a otros agentes
- El LLM es el único punto de decisión

### 2. Múltiples Herramientas

- El agente puede usar **muchas tools**
- Pero es **un solo agente** el que decide cuándo y cómo usarlas
- Tools son extensiones, no agentes independientes

### 3. Contexto Unificado

- **Toda la información** está en un solo contexto
- No hay necesidad de "sincronizar" entre agentes
- Coherencia automática

### 4. State Centralizado

- Un solo state machine
- Fácil de rastrear y debuggear
- No hay problemas de consistencia distribuida

In [2]:
# Ejemplo conceptual de Single-Agent

class SingleAgent:
    """
    Arquitectura Single-Agent simple
    """
    def __init__(self, llm, tools):
        self.llm = llm              # UN SOLO LLM
        self.tools = tools          # MÚLTIPLES TOOLS
        self.context = []           # CONTEXTO UNIFICADO
        self.state = {}             # STATE CENTRALIZADO

    def run(self, task):
        """
        Loop simple: el mismo agente decide todo
        """
        while not self.is_done():
            # EL MISMO LLM razona sobre todo
            decision = self.llm.decide(self.context, task)

            # EL MISMO AGENTE ejecuta la tool
            result = self.execute_tool(decision)

            # TODO se guarda en EL MISMO CONTEXTO
            self.context.append(result)

        return self.generate_response()

print("✅ Single-Agent: UN cerebro, MÚLTIPLES herramientas, CONTEXTO unificado")

✅ Single-Agent: UN cerebro, MÚLTIPLES herramientas, CONTEXTO unificado


## Cuándo Usar Single-Agent

### ✅ Usar Single-Agent cuando:

#### 1. Contexto Cohesivo
**La tarea requiere contexto compartido constante**

- Conversaciones continuas
- Análisis de documentos largos
- Tareas que requieren "recordar" todo el historial

**Ejemplo:** Asistente personal que necesita conocer todas tus preferencias

#### 2. Budget Limitado
**Minimizar costos de LLM**

- Cada llamada a LLM cuesta dinero
- Single-Agent = 1 llamada por iteración
- Multi-Agent = N llamadas por iteración

**Cálculo:** Si Multi-Agent usa 5 agentes → 5x el costo

#### 3. Latencia Crítica
**Respuestas deben ser rápidas (<1s)**

- Menos overhead de coordinación
- Sin esperas entre agentes
- Camino directo: input → LLM → output

**Ejemplo:** Chatbot de atención al cliente en tiempo real

#### 4. Debugging Prioritario
**Necesitas entender exactamente qué pasó**

- Un solo punto de fallo
- Una sola traza de ejecución
- Fácil reproducir problemas

**Ejemplo:** Sistema crítico de salud donde cada decisión debe ser auditable

## Ejemplos Concretos

### Ejemplo 1: Asistente Personal

**Caso de uso:** Gestionar calendario, emails, tareas

**Por qué Single-Agent:**
- ✅ Necesita contexto de todas tus preferencias
- ✅ Debe recordar conversaciones previas
- ✅ Las tareas están interconectadas (email → tarea → calendario)

```
Usuario: "Agenda reunión con Juan el martes"
         ↓
    Single Agent:
    1. Revisa calendario (tool)
    2. Encuentra slot disponible
    3. Crea evento
    4. Envía email a Juan (tool)
    5. Responde confirmación
    
TODO en el mismo contexto.
```

### Ejemplo 2: Coding Agent

**Caso de uso:** Escribir, debuggear, testear código

**Por qué Single-Agent:**
- ✅ Debe entender TODO el contexto del código
- ✅ Debugging requiere ver todo el historial
- ✅ Cambios están interconectados

```
Usuario: "Este código tiene un bug"
         ↓
    Single Agent:
    1. Lee código completo
    2. Identifica bug
    3. Genera fix
    4. Ejecuta tests (tool)
    5. Verifica que funciona
    
El MISMO agente ve todo el proceso.
```

### Ejemplo 3: Research Assistant

**Caso de uso:** Investigar tema y crear reporte

**Por qué Single-Agent:**
- ✅ Debe sintetizar información de múltiples fuentes
- ✅ El reporte requiere coherencia narrativa
- ✅ Búsquedas iterativas basadas en hallazgos previos

```
Usuario: "Investiga impacto de IA en educación"
         ↓
    Single Agent:
    1. Búsqueda inicial (tool)
    2. Identifica sub-temas
    3. Búsquedas específicas (tool)
    4. Sintetiza hallazgos
    5. Escribe reporte coherente
    
Coherencia narrativa = Single-Agent.
```

## Limitaciones y Escalado

### Limitaciones de Single-Agent

#### 1. Context Window Finito
**Problema:** Solo cabe N tokens en el contexto

- Conversaciones largas → contexto se llena
- Documentos grandes → no caben completos
- Tareas complejas → demasiada información

**Mitigación:**
- Summarization progresiva
- RAG (Retrieval-Augmented Generation)
- Sliding window

#### 2. Sin Especialización
**Problema:** Un solo agente no puede ser experto en todo

- LLM general vs LLMs especializados
- Menos profundidad en dominios específicos

**Cuándo es un problema:**
- Necesitas expertise muy especializada (legal, médica)
- Múltiples dominios completamente diferentes

#### 3. Sin Paralelización Real
**Problema:** No puede hacer dos cosas a la vez

- Ejecución secuencial
- No aprovecha paralelización

**Impacto:**
- Tareas largas → más tiempo total
- No puede distribuir carga

#### 4. Single Point of Failure
**Problema:** Si el agente falla, todo falla

- No hay redundancia
- No hay fallback automático

### Cuándo Escalar a Multi-Agent

Considera Multi-Agent cuando:

```
┌─────────────────────────────────────────┐
│  SEÑALES para pasar a Multi-Agent:     │
├─────────────────────────────────────────┤
│                                         │
│ 🚨 Context overflow constante           │
│ 🚨 Necesitas especialización profunda   │
│ 🚨 Paralelización daría gran beneficio  │
│ 🚨 Sub-tareas completamente separables  │
│ 🚨 Debugging ya no es prioridad         │
│ 🚨 Budget no es limitación              │
│                                         │
└─────────────────────────────────────────┘
```

---
# 3.3 Multi-Agent Architectures

**Multi-Agent:** Múltiples agentes independientes que colaboran para lograr un objetivo común.

```
┌─────────────────────────────────────────┐
│       MULTI-AGENT SYSTEM                │
└─────────────────────────────────────────┘
                    │
                    │ Task
                    ▼
         ┌──────────────────────┐
         │   ORCHESTRATOR       │
         │  (puede ser agente)  │
         └──────────┬───────────┘
                    │
        ┌───────────┼───────────┐
        │           │           │
        ▼           ▼           ▼
    ┌───────┐   ┌───────┐   ┌───────┐
    │Agent 1│   │Agent 2│   │Agent 3│ ← MÚLTIPLES CEREBROS
    │       │   │       │   │       │
    │LLM +  │   │LLM +  │   │LLM +  │
    │Tools  │   │Tools  │   │Tools  │
    └───────┘   └───────┘   └───────┘
        │           │           │
        └───────────┴───────────┘
                    │
                    ▼
            Result Aggregation
```

**Diferencia clave:** Cada agente tiene su propio LLM y puede razonar independientemente.

# 3.3.1 Peer-to-Peer (Democratic)

## Descripción

**Peer-to-Peer:** Agentes con **igual autoridad** que se comunican **directamente** entre sí.

```
          Agent A ←──────→ Agent B
             ↕      ⤫      ↕
             ↕        ⤫    ↕
          Agent C ←──────→ Agent D

• Sin jerarquía
• Comunicación directa entre cualquier par
• Decisiones por consenso/voting
```

## Características

### 1. Sin Jerarquía
- **Todos los agentes** tienen igual peso
- No hay "jefe" o "manager"
- Cada agente puede iniciar comunicación

### 2. Comunicación Directa
- Los agentes se hablan **entre sí**
- No necesitan intermediario
- Cada mensaje puede ir de A → B directamente

### 3. Autonomía Individual
- Cada agente **decide** por sí mismo
- No reciben órdenes de arriba
- Pueden rechazar o proponer alternativas

### 4. Consenso por Voting/Debate
- Decisiones colectivas por:
  - **Voting:** Cada agente vota, mayoría gana
  - **Debate:** Discuten hasta llegar a acuerdo
  - **Combinación:** Debate → Voting final

In [4]:
# Ejemplo conceptual de Peer-to-Peer

class PeerToPeerSystem:
    """
    Sistema donde agentes tienen igual autoridad
    """
    def __init__(self, agents):
        self.agents = agents  # Lista de agentes "iguales"

    def solve_by_voting(self, problem):
        """
        Cada agente propone solución, se vota
        """
        proposals = []

        # Cada agente propone independientemente
        for agent in self.agents:
            proposal = agent.propose_solution(problem)
            proposals.append(proposal)

        # Cada agente vota por todas las propuestas
        votes = {}
        for agent in self.agents:
            vote = agent.vote(proposals)
            votes[vote] = votes.get(vote, 0) + 1

        # Mayoría gana
        winner = max(votes.items(), key=lambda x: x[1])[0]
        return winner

print("✅ Peer-to-Peer: Agentes IGUALES, decisiones por CONSENSO")

✅ Peer-to-Peer: Agentes IGUALES, decisiones por CONSENSO


## Cuándo Usar Peer-to-Peer

### ✅ Usar P2P cuando:

#### 1. Múltiples Perspectivas Valiosas
**Necesitas ver el problema desde varios ángulos**

**Ejemplo:** Análisis de inversión
- Agente 1: Perspectiva técnica (charts, indicadores)
- Agente 2: Perspectiva fundamental (financials, industry)
- Agente 3: Perspectiva de riesgo (volatilidad, correlaciones)

Ninguna perspectiva es "superior" → P2P

#### 2. Verificación Cruzada Necesaria
**Necesitas validar resultados independientemente**

**Ejemplo:** Sistema crítico de salud
- Agente 1: Propone diagnóstico
- Agente 2: Verifica independientemente
- Agente 3: Third opinion

Seguridad > Eficiencia → P2P

#### 3. Debate es Beneficioso
**La discusión mejora la calidad**

**Ejemplo:** Estrategia de negocio
- Agentes debaten pros/cons
- Cuestionan supuestos
- Refinan propuestas mutuamente

El proceso de debate **es** el valor → P2P

## Trade-offs de Peer-to-Peer

### ✅ Ventajas

| Ventaja | Por qué |
|---------|----------|
| **Robustez** | Sin single point of failure, otros agentes compensan |
| **Calidad alta** | Múltiples perspectivas detectan errores |
| **Fairness** | Todas las opiniones cuentan igual |
| **Creatividad** | Debate genera ideas nuevas |

### ❌ Desventajas

| Desventaja | Por qué |
|------------|----------|
| **Costo alto** | N agentes = N llamadas LLM por ronda |
| **Latencia alta** | Múltiples rondas de comunicación |
| **Puede no converger** | Agentes pueden no llegar a acuerdo |
| **Overhead de coordinación** | Sincronizar N agentes es complejo |

### Cuándo NO usar P2P

```
❌ Budget limitado
❌ Latencia crítica (<2s)
❌ Tarea tiene jerarquía natural
❌ Un agente claramente superior
❌ Decisiones simples (overkill)
```

# 3.3.2 Hierarchical (Manager-Worker)

## Descripción

**Hierarchical:** Un agente **Manager** coordina y delega a múltiples agentes **Workers**.

```
                ┌──────────────┐
                │   ORCHEST    │
                │              │
                │ • Coordina   │
                │ • Delega     │
                │ • Agrega     │
                └──────┬───────┘
                       │
         ┌─────────────┼─────────────┐
         │             │             │
         ▼             ▼             ▼
    ┌────────┐    ┌────────┐    ┌────────┐
    │Worker 1│    │Worker 2│    │Worker 3│
    │        │    │        │    │        │
    │Ejecuta │    │Ejecuta │    │Ejecuta │
    │tarea A │    │tarea B │    │tarea C │
    └────────┘    └────────┘    └────────┘
         │             │             │
         └─────────────┴─────────────┘
                       │
                       ▼
                Resultados al Manager
```

## Características

### 1. El orquestador coordina

El **orquestador** es responsable de:
- Descomponer tarea en sub-tareas
- Asignar sub-tareas a workers
- Monitorear progreso
- Agregar resultados

**el orquestador NO ejecuta** tareas directamente (solo coordina)

### 2. Workers Ejecutan

Los **Workers** son responsables de:
- Ejecutar la tarea asignada
- Reportar resultado al Manager
- Solicitar ayuda si es necesario

**Workers NO coordinan** entre sí (solo con Manager)

### 3. Comunicación Centralizada

```
Workers NO se hablan entre sí:
Worker A ✗ Worker B  (no permitido)

Todo pasa por orquestador:
Worker A → Manager → Worker B  (permitido)
```

### 4. Escalabilidad Horizontal

- Fácil añadir más workers
- Manager coordina N workers sin cambio
- Paralelización natural

In [5]:
# Ejemplo conceptual de Hierarchical

class HierarchicalSystem:
    """
    Sistema Manager-Worker
    """
    def __init__(self, orchestrator, workers):
        self.orchestrator = orchestrator
        self.workers = workers

    def solve(self, task):
        """
        Manager descompone y delega
        """
        # Manager descompone tarea
        subtasks = self.manager.decompose(task)

        # Manager asigna a workers
        results = []
        for subtask, worker in zip(subtasks, self.workers):
            result = worker.execute(subtask)
            results.append(result)

        # Manager agrega resultados
        final_result = self.manager.aggregate(results)
        return final_result

print("✅ Hierarchical: MANAGER coordina, WORKERS ejecutan")

✅ Hierarchical: MANAGER coordina, WORKERS ejecutan


## Cuándo Usar Hierarchical

### ✅ Usar Hierarchical cuando:

#### 1. Tareas Descomponibles
**La tarea se divide claramente en sub-tareas independientes**

**Ejemplo:** Generar reporte financiero
- Manager: "Necesito reporte Q4"
- Worker 1: Analiza revenue
- Worker 2: Analiza costs
- Worker 3: Analiza profit margins
- Manager: Agrega en reporte final

#### 2. Especialización por Rol
**Diferentes workers son expertos en diferentes áreas**

**Ejemplo:** Sistema de customer support
- Manager: Clasifica tipo de query
- Worker "Billing": Maneja temas de facturación
- Worker "Technical": Maneja temas técnicos
- Worker "Sales": Maneja pre-sales

#### 3. Control Centralizado Necesario
**Necesitas un punto central de decisión y auditoría**

**Ejemplo:** Sistema regulado
- Manager: Verifica compliance
- Manager: Audita todas las decisiones
- Manager: Tiene veto power

## Trade-offs de Hierarchical

### ✅ Ventajas

| Ventaja | Por qué |
|---------|----------|
| **Escalabilidad** | Fácil añadir workers sin cambiar arquitectura |
| **Especialización** | Workers se enfocan en su dominio |
| **Paralelización** | Workers ejecutan en paralelo |
| **Simplicidad de coordinación** | Manager centraliza decisiones |
| **Control claro** | Responsabilidades bien definidas |

### ❌ Desventajas

| Desventaja | Por qué |
|------------|----------|
| **Manager bottleneck** | Todo pasa por Manager → puede ser lento |
| **Single point of failure** | Si Manager falla, todo falla |
| **Overhead de Manager** | Manager consume tokens sin ejecutar |
| **Workers no colaboran** | Pierdes sinergia entre workers |

### Cuándo NO usar Hierarchical

```
❌ Tareas no descomponibles
❌ Workers necesitan colaborar entre sí
❌ No hay jerarquía natural
❌ Manager se convierte en bottleneck
```

# 3.3.3 Hybrid Architectures

**Hybrid:** Combinaciones de arquitecturas para casos de uso específicos.

## 1. Cloud-Edge Hybrid

**Concepto:** Razonamiento en cloud, ejecución en edge

```
       ☁️ CLOUD
    ┌─────────────┐
    │ Reasoning   │ ← LLM grande y capaz
    │ Agent       │   Toma decisiones
    │ (GPT-4)     │
    └──────┬──────┘
           │
           │ Plan/Instrucciones
           │
           ▼
       📱 EDGE
    ┌─────────────┐
    │ Execution   │ ← Modelo pequeño
    │ Agent       │   Ejecuta rápido
    │ (Llama 3)   │   Sin latencia
    └─────────────┘
```

**Uso:** IoT, mobile apps, sistemas con conectividad intermitente

## 2. Specialized-General Hybrid

**Concepto:** Agentes especializados + fallback generalista

```
  Query
    │
    ▼
 Router
    │
    ├──→ Specialist A (si dominio A)
    ├──→ Specialist B (si dominio B)
    └──→ Generalist (si ambiguo o fuera de dominio)
```

**Uso:** Customer support con dominios claros pero queries variables

## 3. Layered Hybrid

**Concepto:** Múltiples capas de abstracción

```
Layer 3: Strategic (Planning)
           │
           ▼
Layer 2: Tactical (Coordination)
           │
           ▼
Layer 1: Operational (Execution)
```

**Uso:** Sistemas complejos con múltiples niveles de decisión

---
# 3.4 Trade-Offs Arquitectónicos Fundamentales

## Tabla Comparativa Completa

No existe la arquitectura "perfecta" - solo trade-offs.

## Análisis Detallado de Trade-offs

### 1. Coherencia

**Single-Agent: ⭐⭐⭐**
- Un solo contexto → coherencia automática
- No hay contradicciones entre agentes

**Multi-Agent: ⭐⭐**
- Múltiples contextos → necesitas sincronizar
- Agentes pueden tener info inconsistente

### 2. Costo

**Single-Agent: ⭐⭐⭐**
- 1 llamada LLM por iteración
- Costo = N iteraciones × 1 llamada

**Hierarchical: ⭐⭐**
- 1 (manager) + N (workers) llamadas
- Costo = N iteraciones × (1 + N) llamadas

**Peer-to-Peer: ❌**
- N agentes × M rondas de debate
- Costo = N × M llamadas (el más alto)

### 3. Latencia

**Single-Agent: ⭐⭐⭐**
- Secuencial pero sin overhead
- Latencia = T(reasoning) + T(tool)

**Hierarchical: ⭐⭐**
- Workers en paralelo → reduce latencia
- Pero Manager añade overhead

**Peer-to-Peer: ❌**
- Múltiples rondas de comunicación
- Latencia = N rondas × T(agente)

### 4. Especialización

**Single-Agent: ⭐**
- Un LLM general para todo
- No hay expertise profunda

**Multi-Agent: ⭐⭐⭐**
- Cada agente es experto en su dominio
- Puedes usar modelos especializados

### 5. Debugging

**Single-Agent: ⭐⭐⭐**
- Una traza lineal
- Fácil reproducir problemas

**Multi-Agent: ⭐**
- Múltiples trazas concurrentes
- Problemas de timing y sincronización

### 6. Escalabilidad

**Single-Agent: ⭐**
- Context window es el límite
- No puedes añadir más "capacidad"

**Hierarchical: ⭐⭐⭐**
- Añadir workers es fácil
- Escala horizontalmente

### 7. Paralelización

**Single-Agent: ❌**
- Inherentemente secuencial
- Un solo LLM → no paraleliza

**Hierarchical: ⭐⭐⭐**
- Workers ejecutan en paralelo
- Reduce tiempo total significativamente

### 8. Robustez

**Single-Agent: ⭐**
- Si falla, todo falla
- No hay redundancia

**Peer-to-Peer: ⭐⭐⭐**
- Múltiples agentes verifican
- Si uno falla, otros compensan

## Criterios de Decisión

### Cuándo Single-Agent

```python
if (
    coherencia_crítica or
    budget_limitado or
    latencia_crítica or
    debugging_prioritario or
    tarea_simple_o_mediana
):
    return "Single-Agent"
```

### Cuándo Multi-Agent

```python
if (
    especialización_necesaria or
    paralelización_beneficiosa or
    tareas_claramente_separables or
    múltiples_perspectivas_valiosas or
    escalabilidad_requerida
):
    # Decide qué tipo de Multi-Agent
    if múltiples_perspectivas_valiosas:
        return "Peer-to-Peer"
    elif tareas_descomponibles:
        return "Hierarchical"
    else:
        return "Hybrid"
```

## Principio Fundamental

> **No hay arquitectura "mejor" - solo trade-offs apropiados para tu caso de uso.**

### Preguntas para decidir:

1. **¿Qué es más importante?**
   - ⏱️ Velocidad → Single-Agent
   - 💰 Costo → Single-Agent
   - 🎯 Calidad → Multi-Agent (P2P)
   - 📈 Escala → Multi-Agent (Hierarchical)

2. **¿Cuál es tu constraint principal?**
   - Budget → Single-Agent
   - Context window → Multi-Agent
   - Latencia → Single-Agent o Hierarchical

3. **¿Tu tarea es descomponible?**
   - Sí → Hierarchical
   - No → Single-Agent o P2P

4. **¿Necesitas múltiples perspectivas?**
   - Sí → Peer-to-Peer
   - No → Single-Agent o Hierarchical

### Matriz de Decisión Final

In [ ]:
# Matriz de decisión simplificada
decision_matrix = pd.DataFrame({
    'Si tu prioridad es...': [
        'Bajo costo',
        'Baja latencia',
        'Fácil debugging',
        'Alta calidad',
        'Especialización',
        'Escalabilidad',
        'Robustez',
        'Paralelización'
    ],
    'Elige...': [
        'Single-Agent',
        'Single-Agent',
        'Single-Agent',
        'Peer-to-Peer',
        'Multi-Agent (cualquiera)',
        'Hierarchical',
        'Peer-to-Peer',
        'Hierarchical'
    ],
    'Por qué': [
        '1 llamada LLM por step',
        'Sin overhead de coordinación',
        'Una sola traza',
        'Verificación cruzada múltiple',
        'Agentes expertos por dominio',
        'Fácil añadir workers',
        'Sin single point of failure',
        'Workers en paralelo'
    ]
})

display(decision_matrix)

---
# 📝 Resumen del Tema 3

## Puntos Clave

### 3.1 ¿Qué es Arquitectura?
- ✅ Arquitectura = decisiones **estructurales**
- ✅ 3 dimensiones: Arquitectura, Flujo, Capacidades
- ✅ Arquitectura como **restricción habilitadora**

### 3.2 Single-Agent
- ✅ Un cerebro, múltiples tools, contexto unificado
- ✅ Ideal para: coherencia, bajo costo, baja latencia
- ✅ Limitado por: context window, sin especialización

### 3.3 Multi-Agent
- ✅ **Peer-to-Peer:** Agentes iguales, consenso por voting/debate
- ✅ **Hierarchical:** Manager coordina, Workers ejecutan
- ✅ **Hybrid:** Combinaciones para casos específicos

### 3.4 Trade-offs
- ✅ No hay arquitectura "mejor"
- ✅ Single-Agent: coherencia + costo vs especialización
- ✅ Multi-Agent: calidad + escala vs complejidad + costo

